# Name That LLM — checkpoint 3 November 2026

Laporan yang dapat dijalankan ulang: EDA, audit validasi/cache, eksperimen CPU,
ablation, serta ekspektasi versus skor submission resmi. Target riset 0,99 belum
tercapai. Snapshot skor resmi terbaik akun adalah **0,665**. Notebook ini memakai
hasil eksperimen yang sudah disimpan; tidak membutuhkan GPU atau membuat submission.

Jalankan dari root snapshot source repo. Raw train tidak didistribusikan lewat GitHub;
letakkan data yang diunduh secara sah di `data/` atau attach competition di Kaggle.

In [1]:
import json, time
from contextlib import contextmanager
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in candidates if (p/'research/experiments/e15_reader_foldlocal/results.json').exists()), None)
assert ROOT is not None, 'Jalankan dari root source snapshot atau folder notebook di dalamnya.'
@contextmanager
def step(name, eta_seconds):
    begin = time.perf_counter()
    print(f'{name} | estimasi {eta_seconds}s', flush=True)
    yield
    print(f'{name} | selesai {time.perf_counter()-begin:.2f}s', flush=True)

## 1. EDA: unit observasi dan generalisasi
Split harus memisahkan prompt. Angka 70/15/15 diterapkan pada prompt dan
dibulatkan menjadi 105/22/23 dari 150 prompt. Lima blok test mencakup 115 prompt,
bukan 150; ini lima rotasi holdout, bukan test coverage 100% dari 5-fold biasa.

In [2]:
with step('EDA', 2):
    local = ROOT/'data/train.csv'
    kaggle = Path('/kaggle/input/competitions/name-that-llm/train.csv')
    path = local if local.exists() else kaggle
    if path.exists():
        train = pd.read_csv(path)
        assert {'id','text','model','prompt_id'}.issubset(train.columns)
        assert train.id.is_unique and train.notna().all().all()
        print('rows',len(train),'prompts',train.prompt_id.nunique(),'labels',train.model.nunique())
        display(train.model.value_counts().rename('support').to_frame())
        print('words:',train.text.str.split().str.len().describe().round(2).to_dict())
        markers = pd.DataFrame({k: train.text.str.contains(v, regex=False)
                                for k,v in {'em_dash':'—','curly_apostrophe':'’','semicolon':';'}.items()})
        markers['model'] = train.model
        display(markers.groupby('model').mean().round(3))
    else:
        print('Raw train belum dipasang. Ringkasan audit:2699 rows,150 prompts,18 labels;149/150 per kelas.')

EDA | estimasi 2s
rows 2699 prompts 150 labels 18
words: {'count': 2699.0, 'mean': 114.66, 'std': 58.28, 'min': 36.0, '25%': 64.0, '50%': 101.0, '75%': 157.0, 'max': 274.0}
EDA | selesai 0.27s


,support
model,
qwen3.8-max,150
claude-sonnet-5,150
gemini-3.6-flash,150
gpt-5.6-terra,150
glm-5.3-flash,150
gemini-3.1-pro-preview,150
gemini-3.8-flash,150
claude-opus-4-6,150
gpt-5.6-luna,150


,em_dash,curly_apostrophe,semicolon
model,,,
claude-fable-5,0.832,0.000,0.336
claude-opus-4-6,0.753,0.000,0.153
claude-opus-4-8,0.627,0.000,0.240
claude-opus-5,0.813,0.000,0.540
claude-sonnet-5,0.733,0.000,0.233
gemini-3.1-pro-preview,0.000,0.060,0.193
gemini-3.6-flash,0.000,0.193,0.227
gemini-3.7-flash,0.007,0.307,0.233
gemini-3.8-flash,0.000,0.247,0.280


## 2. Audit yang mengubah interpretasi skor

- OOF LoRA lama dipindahkan ke rotasi 70/15/15. Setiap baris tetap out-of-prompt,
  tetapi proses stacking bukan nested training yang sepenuhnya terpisah.
- Standardisasi/PCA lama dilatih global. E15/E16 melatih transformasi hanya di train.
- Pool yang direkonstruksi tidak cocok urutan labelnya dengan cache lama (91,33%
  posisi berbeda). Ini bukan bukti label cache salah. E15/E16 memakai fitur host
  saja; provenance pool harus dipulihkan sebelum dipakai lagi.
- Tidak ada hasil hidden per kelas yang tersedia. Diagnostik per kelas di sini
  berasal dari gold lokal, sedangkan leaderboard hanya metrik agregat.

In [3]:
with step('muat manifest', 1):
    manifests={}
    for name in ['e15_reader_foldlocal','e16_topic_projection']:
        m=json.loads((ROOT/'research/experiments'/name/'manifest.json').read_text())
        manifests[name]=m
        print(name,'signature',m['signature'],'split',m['split_md5'])
    assert manifests['e15_reader_foldlocal']['split_md5']==manifests['e16_topic_projection']['split_md5']
    print('External pool used in E15: False; alignment audit:',manifests['e15_reader_foldlocal']['reconstructed_pool_alignment'])

muat manifest | estimasi 1s
e15_reader_foldlocal signature 5ff79f4bfd9cc871e27f58c37490a9e8276b21824cb41d744822b561163465d2 split e52bf922570b1d00a0e51c510ac4f16d
e16_topic_projection signature cdb0a6989fb3914484b666d0885a487c864ad17d9dd104c616d4cb2e527d9d65 split e52bf922570b1d00a0e51c510ac4f16d
External pool used in E15: False; alignment audit: False
muat manifest | selesai 0.00s


## 3. E15: mengganti scorer pembaca frozen
Uji Qwen8/Qwen27 layer16/27, train-only scaler/PCA128, LR dua C dan shrinkage-LDA.
Pilih kandidat hanya pada validation. Anchor fixed:Qwen8-layer16 LR C0,003.
Ini perbandingan host-only; tidak setara dengan ensemble publik yang memakai pool besar.

In [4]:
with step('E15 hasil dan log', 1):
    e15=json.loads((ROOT/'research/experiments/e15_reader_foldlocal/results.json').read_text())
    folds=json.loads((ROOT/'research/experiments/e15_reader_foldlocal/folds.json').read_text())
    display(pd.DataFrame([dict(fold=r['fold'],model=r['chosen']['name'],train=r['chosen']['train_ba'],
                              val=r['chosen']['validation_ba'],test=r['selected_test_ba'],
                              anchor=r['anchor_test_ba'],seconds=r['elapsed_s']) for r in folds]).round(4))
    print('mean test',e15['selected_test_ba_mean'],'paired delta CI95',e15['paired_delta_ci95'])
    print('Keputusan:belum terkonfirmasi; tidak memenuhi submission gate.')

E15 hasil dan log | estimasi 1s
mean test 0.3546113306982872 paired delta CI95 [-0.011111111111111127, 0.0338675018956908]
Keputusan:belum terkonfirmasi; tidak memenuhi submission gate.
E15 hasil dan log | selesai 0.01s


,fold,model,train,val,test,anchor,seconds
0,0,q27_l16_lr_C0.003,0.6053,0.3917,0.3502,0.3333,2.1908
1,1,q27_l16_lr_C0.003,0.5984,0.3712,0.4059,0.3866,2.0778
2,2,q27_l16_lr_C0.003,0.6176,0.3737,0.3430,0.3237,2.0587
3,3,q27_l16_lr_C0.003,0.6187,0.3535,0.3937,0.3599,1.9542
4,4,q8_l27_lda_auto,0.4943,0.3510,0.2802,0.3140,2.0174


## 4. E16 dan E17: fusion versus proyeksi topik
E16 mempelajari arah nuisance dari centroid prompt training dalam PCA space,
kemudian memilih rank/gamma/C dan kekuatan fusion dengan SVC di validation.
E17 memisahkan manfaat reader fusion dari manfaat proyeksi topik itu sendiri.
Cached SVC lolos cek shape, finite validation/test dan NaN training mask;
provenance lengkap cache lama tetap lebih lemah daripada training baru dari manifest.

In [5]:
with step('E16/E17 ablation', 1):
    e16=json.loads((ROOT/'research/experiments/e16_topic_projection/results.json').read_text())
    e17=json.loads((ROOT/'research/experiments/e17_topic_ablation/results.json').read_text())
    display(pd.DataFrame({'test_mean':e17['test_mean'],'test_pooled':e17['test_pooled']}).round(4))
    display(pd.DataFrame([dict(comparison=k,ci95_low=v[0],ci95_high=v[1])
                          for k,v in e17['paired_ci95'].items()]).round(4))
    print('E16 positive rotations',e16['positive_rotations'],'/5')
    print('CPU seconds:E15',round(e15['elapsed_s'],2),'E16',round(e16['elapsed_s'],2),'E17',round(e17['elapsed_s'],2))
    print('Fusion membantu SVC; efek tambahan proyeksi belum terkonfirmasi jika CI melintasi0.')
    print('Skor cabang ini tidak mengalahkan champion resmi0.665; tidak membuat submission.')

E16/E17 ablation | estimasi 1s
E16 positive rotations 5 /5
CPU seconds:E15 13.28 E16 9.92 E17 4.8
Fusion membantu SVC; efek tambahan proyeksi belum terkonfirmasi jika CI melintasi0.
Skor cabang ini tidak mengalahkan champion resmi0.665; tidak membuat submission.
E16/E17 ablation | selesai 0.01s


,test_mean,test_pooled
svc,0.5006,0.5006
fusion_only,0.5223,0.5223
e16_selected,0.5242,0.5243


,comparison,ci95_low,ci95_high
0,fusion_only minus svc,0.0101,0.0333
1,e16_selected minus fusion_only,-0.0014,0.0053


## 5. E14 selesai dan E18 analisis kesalahan
E14 dibandingkan hanya pada fold0–2 terhadap kontrol yang identik, tanpa
membandingkan rerata3fold dengan5fold. CI bersyarat pada model historis dan tidak
mencakup bias seluruh proses research. Hasil E16 adalah cabang SVC+reader.

In [6]:
with step('audit E14 dan error slices', 1):
    audit=json.loads((ROOT/'research/experiments/e18_closing_audit/results.json').read_text())
    display(pd.DataFrame([dict(model=k,control=v['control_mean'],e14=v['e14_mean'],
                              ci_low=v['paired_prompt_ci95'][0],ci_high=v['paired_prompt_ci95'][1],
                              positive_folds=v['positive_folds']) for k,v in audit['e14'].items()]).round(4))
    print('E14 COMPLETE. OOF/log diunduh; source tidak menyimpan bobot tahap2. Tidak dipromosikan.')
    display(pd.DataFrame(audit['e16']['per_class']).round(4))
    display(pd.DataFrame(audit['e16']['length_slices']).round(4))
    display(pd.DataFrame(audit['e16']['top_confusions'][:6]))
    print('Fixes',audit['e16']['fixes'],'breaks',audit['e16']['breaks'])
    print('Family accuracy',audit['e16']['family_accuracy'],
          'fraction errors within family',audit['e16']['fraction_errors_within_family'])
    print('Slice adalah diagnostik pasca-test; bukan izin menyesuaikan kandidat pada test.')

audit E14 dan error slices | estimasi 1s
E14 COMPLETE. OOF/log diunduh; source tidak menyimpan bobot tahap2. Tidak dipromosikan.
Fixes 107 breaks 58
Family accuracy 0.7800869985500242 fraction errors within family 0.5376016260162602
Slice adalah diagnostik pasca-test; bukan izin menyesuaikan kandidat pada test.
audit E14 dan error slices | selesai 0.04s


,model,control,e14,ci_low,ci_high,positive_folds
0,q8,0.5686,0.5649,-0.0198,0.0130,1
1,ll8,0.5792,0.5705,-0.0265,0.0086,1


,model,n,svc_recall,e16_recall
0,claude-fable-5,114,0.2281,0.2281
1,claude-opus-4-6,115,0.7217,0.7826
2,claude-opus-4-8,115,0.3826,0.4696
3,claude-opus-5,115,0.6522,0.7130
4,claude-sonnet-5,115,0.6696,0.6609
5,gemini-3.1-pro-preview,115,0.8087,0.8174
6,gemini-3.6-flash,115,0.7217,0.6957
7,gemini-3.7-flash,115,0.1826,0.2261
8,gemini-3.8-flash,115,0.6783,0.6957
9,glm-5.3,115,0.2348,0.2783


,words,n,classes,svc_accuracy,e16_accuracy,e16_BA_present_classes
0,"(0.0, 64.0]",532,18,0.4041,0.4211,0.4098
1,"(64.0, 100.0]",504,18,0.4821,0.5159,0.5152
2,"(100.0, 160.0]",547,18,0.5137,0.5375,0.5341
3,"(160.0, inf]",486,18,0.6111,0.6317,0.6359


,true,predicted,n
0,gemini-3.7-flash,gemini-3.8-flash,68
1,gpt-5.6-sol,gpt-5.6-terra,26
2,kimi-k3,gpt-5.5,21
3,claude-fable-5,glm-5.3,21
4,glm-5.3-flash,glm-5.3,20
5,glm-5.3,glm-5.3-flash,20


## 6. Ekspektasi versus leaderboard resmi
Judul seperti “0.75+” bukan pengukuran. Jangan menghitung gap CV-LB jika CV yang
sebanding tidak tersedia. Skor private masih tidak diketahui. Data di bawah snapshot,
bukan polling live; cek ulang API sebelum tindakan baru.

In [7]:
with step('riwayat resmi', 1):
    history=json.loads((ROOT/'research/live/20261103/submissions.txt').read_text())
    rows=[]
    for r in history:
        rows.append(dict(submission=r['ref'],date=r['date'],status=r['status'].split('.')[-1],
                         public_BA=float(r['publicScore']) if r.get('publicScore') else None,
                         description=r['description']))
    history_table=pd.DataFrame(rows)
    display(history_table)
    print('Official best',history_table.public_BA.max())
    for ref,context in [(57889656,'target judul0.75+, tidak ada CV0.75 terverifikasi'),
                        (57919689,'penambahan komponen:LB0.661'),
                        (57919873,'perubahan multi-komponen:LB0.663')]:
        hit=history_table[history_table.submission==ref]
        if len(hit): print(ref,context,'actual',hit.public_BA.iloc[0])

riwayat resmi | estimasi 1s
Official best 0.665
57889656 target judul0.75+, tidak ada CV0.75 terverifikasi actual 0.639
57919689 penambahan komponen:LB0.661 actual 0.661
57919873 perubahan multi-komponen:LB0.663 actual 0.663
riwayat resmi | selesai 0.03s


,submission,date,status,public_BA,description
0,57930534,2026-11-03T03:11:20.083000,COMPLETE,0.055,NTL Apex-III Grandmaster SOTA (v3): 8-Model (q...
1,57919873,2026-11-02T19:46:02.997000,COMPLETE,0.663,NTL Apex-II Grandmaster SOTA (v2): Dual-DeBERT...
2,57919689,2026-11-02T19:37:07.900000,COMPLETE,0.661,Sovereign Grandmaster v2 (v3 Exception-Proof):...
3,57907202,2026-11-02T11:45:38.710000,COMPLETE,0.665,NTL Apex I SOTA Contender (190.5k Clean OpenPo...
4,57901383,2026-11-02T08:18:38.430000,COMPLETE,NaN,Sovereign Grandmaster v2: Restored Llama-3.1-8...
5,57889656,2026-11-02T00:47:10.843000,COMPLETE,0.639,SOTA Apex Grandmaster 0.75+ (198k Clean Pool +...
6,57876141,2026-11-01T15:25:29.167000,COMPLETE,0.664,fork of public One Swap 0.664 (zbigniewkirchne...
7,57874130,2026-11-01T14:10:06.747000,COMPLETE,0.659,SOTA Apex 0.72+ Full Multi-Model Ensemble With...
8,57873164,2026-11-01T13:34:23.267000,COMPLETE,0.330,S12 SOTA Apex Ensemble 0.72431 (v2 True Fixed ...
9,57872845,2026-11-01T13:22:41.957000,COMPLETE,NaN,S12 SOTA Apex Ensemble 0.72431 (Fixed Hidden R...


## 7. Komputasi, checkpoint, dan keputusan berikut

GPU akun yang sama habis; TPU masih tersedia pada snapshot. Sesi ini melakukan
eksperimen CPU singkat tanpa membuat runtime GPU/TPU interaktif baru. E14 telah
berjalan sebelum sesi ini; kini COMPLETE dan hasil/log sudah diunduh dan diaudit.
Jangan menggandakan job, menghentikan scoring resmi, atau beralih akun untuk melewati quota.

Berikutnya:replikasi fusion pada cache yang memiliki IDs/revision lengkap, uji pada
ensemble kuat dengan training nested yang benar, dan peroleh gold bertopik baru
untuk konfirmasi. Data exact-source bergaya host bernilai lebih daripada menambah
postprocessing kluster yang sudah gagal. Target0.99 masih belum didukung bukti.

In [8]:
with step('quota snapshot', 1):
    quota=json.loads((ROOT/'research/live/20261103/quota.txt').read_text())
    display(pd.DataFrame(quota))
    print('Handoff:readme.md; logs:research/experiments/e15_reader_foldlocal.log dan e16/e17.')
    print('Jalankan ulang training eksperimen hanya jika dibutuhkan; hasil selesai sudah disimpan.')

quota snapshot | estimasi 1s
Handoff:readme.md; logs:research/experiments/e15_reader_foldlocal.log dan e16/e17.
Jalankan ulang training eksperimen hanya jika dibutuhkan; hasil selesai sudah disimpan.
quota snapshot | selesai 0.01s


,resource,used,remaining,total,refreshAt
0,GPU,31.85h,0.00h,30.00h,2026-11-07T00:00:00+00:00
1,TPU,0.00h,20.00h,20.00h,2026-11-07T00:00:00+00:00


## 8. E19: fusion pada pembanding external-stage1 + SVC
Scope adaptasi tidak mencampur stage2 OOF dengan rotasi. Reader/head memakai
train gold saja, konfigurasi dipilih menurut val logloss, bobot fusion menurut
val BA. Provenance upstream cache historis tetap tidak lengkap. Ini bukan CV
champion leaderboard. Run max_iter400 diarsipkan setelah warning; run final
max_iter2000 mencatat jumlah iterasi dan memverifikasi semua kandidat konvergen.

In [9]:
with step('E19', 1):
    directory=ROOT/'research/experiments/e19_reader_strong_fusion'
    e19=json.loads((directory/'results.json').read_text())
    records=[json.loads((directory/f'fold{f}.json').read_text()) for f in range(5)]
    display(pd.DataFrame([dict(fold=r['fold'],reader=r['chosen']['name'],C=r['chosen']['C'],
        reader_train_BA=r['chosen']['train_ba'],reader_val_BA=r['chosen']['val_ba'],
        ensemble_val_BA=r['selected_val_ba'],baseline_test_BA=r['baseline_test_ba'],
        fusion_test_BA=r['selected_test_ba'],alpha=r['alpha']) for r in records]).round(4))
    print('Mean test',e19['mean_test'],'paired CI95',e19['paired_prompt_ci95'])
    print('Maximum optimization iterations',max(c['iterations'] for r in records for c in r['candidates']))
    print('Elapsed CPU',e19['elapsed_s'],'seconds; no submission.')

E19 | estimasi 1s
Mean test {'baseline': 0.5735177865612648, 'selected': 0.580764163372859, 'candidate': 0.5043697848045674} paired CI95 [-0.003882048651190629, 0.018370306805661495]
Maximum optimization iterations 457
Elapsed CPU 24.736666124983458 seconds; no submission.
E19 | selesai 0.07s


,fold,reader,C,reader_train_BA,reader_val_BA,ensemble_val_BA,baseline_test_BA,fusion_test_BA,alpha
0,0,hybrid_PCA128_logitx3.0,0.003,0.7397,0.5281,0.5587,0.5845,0.6063,0.20
1,1,hybrid_PCA128_logitx3.0,0.003,0.7429,0.4621,0.6086,0.5632,0.5632,0.00
2,2,hybrid_PCA128_logitx3.0,0.003,0.7410,0.5152,0.5960,0.5966,0.5845,0.20
3,3,hybrid_PCA128_logitx3.0,0.003,0.7580,0.4949,0.5783,0.5725,0.5942,0.35
4,4,hybrid_PCA128_logitx3.0,0.003,0.7595,0.4949,0.6035,0.5507,0.5556,0.35


## 9. E20: spectral reader lokal dengan provenance lengkap
Raw response saja, Qwen3-0.6B frozen, DC/first-AC length-normalized, layer8/16/28.
Formula diperiksa terhadap DCT-II scipy; padding tidak masuk trajectory.
Ini adaptasi K=1 tanpa prompt asli; bukan klaim mereplikasi multi-query READER.
Semua transformasi/probe dipelajari dari train tiap rotasi.

In [10]:
with step('E20 spectral', 1):
    directory=ROOT/'research/experiments/e20_spectral_reader'
    e20=json.loads((directory/'results.json').read_text())
    extraction=json.loads((directory/'extraction_status.json').read_text())
    assert extraction['complete']
    feature_manifest=json.loads((directory/'feature_manifest.json').read_text())
    print('Model revision',feature_manifest['model_revision'],'device',feature_manifest['device'])
    print('Extraction',extraction['rows_done'],'rows;',round(extraction['elapsed_s'],2),'seconds')
    print('Truncated rows',feature_manifest['truncated_rows'],'/2699; token cap512')
    records=[json.loads((directory/f'fold{f}.json').read_text()) for f in range(5)]
    display(pd.DataFrame([dict(fold=r['fold'],layer=r['chosen']['DCAC']['layer'],
        C=r['chosen']['DCAC']['C'],reader_train_BA=r['chosen']['DCAC']['train_ba'],
        reader_val_BA=r['chosen']['DCAC']['val_ba'],fusion_val_BA=r['selected_val_ba'],
        **r['test']) for r in records]).round(4))
    print('Mean test',e20['mean_test'],'paired intervals',e20['paired_ci95'])
    print('Head evaluation CPU seconds',e20['elapsed_s'])

E20 spectral | estimasi 1s
Model revision c1899de289a04d12100db370d81485cdf75e47ca device mps
Extraction 2699 rows; 798.02 seconds
Truncated rows 0 /2699; token cap512
Mean test {'baseline': 0.5735177865612648, 'fusion': 0.5792929292929292, 'DC': 0.333816425120773, 'DCAC': 0.3179183135704875} paired intervals {'fusion minus baseline': [-0.005771781506907366, 0.017391304347826087], 'DCAC minus DC': [-0.03382615108375016, 0.002898550724637681]}
Head evaluation CPU seconds 74.39530708300299
E20 spectral | selesai 0.02s


,fold,layer,C,reader_train_BA,reader_val_BA,fusion_val_BA,baseline,fusion,DC,DCAC
0,0,16,0.003,0.9725,0.3232,0.5712,0.5845,0.5966,0.3261,0.3092
1,1,8,0.003,0.9646,0.3283,0.6136,0.5632,0.5752,0.3527,0.3263
2,2,16,0.003,0.9677,0.3434,0.5909,0.5966,0.5821,0.3333,0.3188
3,3,8,0.003,0.9698,0.2854,0.5808,0.5725,0.5918,0.3454,0.3430
4,4,16,0.003,0.9703,0.2929,0.5884,0.5507,0.5507,0.3116,0.2923


## 10. E21: outer5fold dengan cakupan seluruh150prompt
Outer train120prompt, test30prompt. C reader dipilih pada96train/24val di dalam
outer train lalu refit120; alpha0.1 ditetapkan sebelum test. LoRA stage2 memakai
fold yang sesuai, dan skor inner-val dari outer-LoRA tidak dipakai untuk tuning.
Ini protokol terpisah dari lima rotasi70/15/15. Semua prompt tetap data historis
yang sudah pernah ditelaah; tidak berubah menjadi gold eksternal baru.

In [11]:
with step('E21 scope confirmation', 1):
    directory=ROOT/'research/experiments/e21_nested_reader_confirm'
    e21=json.loads((directory/'results.json').read_text())
    assert e21['test_coverage']==1.
    records=[json.loads((directory/f'fold{f}.json').read_text()) for f in range(5)]
    display(pd.DataFrame([dict(fold=r['fold'],C=r['chosen']['C'],
        inner_train_BA=r['chosen']['train_ba'],inner_val_BA=r['chosen']['val_ba'],
        reader_refit_train_BA=r['train_refit_ba'],**r['test']) for r in records]).round(4))
    print('Mean test',e21['mean_test'],'paired CI95',e21['paired_ci95'])
    print('Coverage',e21['test_prompts'],'prompts',e21['test_rows'],'rows; CPU',e21['elapsed_s'],'seconds')
    print(e21['caveat'])

E21 scope confirmation | estimasi 1s
Mean test {'baseline': 0.597088122605364, 'fusion': 0.5923371647509579, 'reader': 0.3193231162196679} paired CI95 [-0.013718617946805865, 0.003718680089485459]
Coverage 150 prompts 2699 rows; CPU 16.183339540992165 seconds
Historical OOF assumed to follow archived worker and original CSV; per-row IDs/model revisions were not recorded. Not independently rerun LoRA. Historical adaptive research means these are not fresh blind gold prompts.
E21 scope confirmation | selesai 0.01s


,fold,C,inner_train_BA,inner_val_BA,reader_refit_train_BA,baseline,reader,fusion
0,0,0.003,0.9751,0.2708,0.9606,0.6204,0.3222,0.6185
1,1,0.003,0.9693,0.3009,0.9574,0.6074,0.3315,0.5833
2,2,0.003,0.9716,0.2847,0.9583,0.5743,0.3133,0.5765
3,3,0.003,0.9820,0.3194,0.9639,0.6019,0.3222,0.5963
4,4,0.003,0.9751,0.3333,0.9583,0.5815,0.3074,0.5870


## 11. E22: kualifikasi data publik baru dengan jumlah kelas terkontrol
Tujuh alias sumber sama persis, namun identitas bobot API tidak immutable.
Max128token menciptakan mismatch dengan paragraf mandiri kompetisi. Sebanyak
50prompt eksternal ditahan dari seluruh fitting. Proxy mengganti pool pada
jumlah dan bobot kelas yang sama; cap500/kelas, bukan full-pool champion.
Recall eksternal pada tujuh label yang hadir tidak setara dengan BA18kelas.

In [12]:
with step('E22 external data', 1):
    directory=ROOT/'research/experiments/e22_reader_data'
    audit=json.loads((directory/'audit.json').read_text())
    e22=json.loads((directory/'results.json').read_text())
    display(pd.DataFrame(audit['per_class']).round(3))
    print('Eligible training',audit['eligible_training_rows'],'heldout external',audit['eligible_external_test_rows'])
    print('Exact pool duplicates',audit['exact_pool'],'near-gold',audit['near_gold_char_ge08'],
          'terminal punctuation fraction',audit['terminal_punctuation_fraction'])
    records=[json.loads((directory/f'proxy_fold{f}.json').read_text()) for f in range(5)]
    display(pd.DataFrame([dict(fold=r['fold'],base_val=r['validation_trials'][0]['validation_ba'],
        replacement_val=r['validation_trials'][1]['validation_ba'],chosen=r['chosen'],
        base_test=r['baseline_test_ba'],selected_test=r['selected_test_ba'],
        external_7label_recall=r['external_present_label_macro_recall']) for r in records]).round(4))
    print('Mean test',e22['mean_test'],'paired CI95',e22['paired_ci95'],'CPU seconds',e22['elapsed_s'])
    print(e22['caveat'])
    print('No new competition submission. Official champion remains0.665 at the saved snapshot.')

E22 external data | estimasi 1s
Eligible training 3033 heldout external 336
Exact pool duplicates 0 near-gold 0 terminal punctuation fraction 0.18114285714285713
Mean test {'baseline': 0.4029863855950812, 'selected': 0.4029863855950812} paired CI95 [0.0, 0.0] CPU seconds 571.1043847910187
Small cap500 proxy, seven exact aliases only. Partial-label external recall is NOT 18-class competition BA, and this is not the leaderboard champion. No inference-ready sparse estimator stored; source/config/fold scores are resumable.
No new competition submission. Official champion remains0.665 at the saved snapshot.
E22 external data | selesai 0.02s


,model,raw,eligible,words_mean
0,claude-opus-4-6,500,497,63.294
1,claude-opus-4-8,500,500,53.156
2,gpt-5.5,500,493,75.836
3,gpt-5.6-luna,500,481,70.266
4,gpt-5.6-sol,500,464,65.838
5,gpt-5.6-terra,500,478,70.782
6,kimi-k3,500,456,71.606


,fold,base_val,replacement_val,chosen,base_test,selected_test,external_7label_recall
0,0,0.4196,0.3818,0,0.4275,0.4275,0.0989
1,1,0.3914,0.3308,0,0.4304,0.4304,0.0841
2,2,0.4040,0.3460,0,0.4082,0.4082,0.0903
3,3,0.4040,0.3535,0,0.3720,0.3720,0.1018
4,4,0.4141,0.3460,0,0.3768,0.3768,0.0959


## 12. Submission V3 selesai0.055: audit jalur gagal
Source original memiliki catch-all pada inferensi utama, lalu dapat menulis
seluruh label sebagai gpt-5.5. Fixture400ID sintetis mereproduksi defect itu
tanpa menjalankan backbone. Skor1/18 konsisten dengan LB mendekati chance,
tetapi log hidden tidak tersedia sehingga kejadian dan exception asli belum
terbukti. Salinan guarded hanya artefak correctness, belum lolos dryrun GPU.

In [13]:
with step('V3 fallback regression', 1):
    failure=json.loads((ROOT/'research/checkpoints/failed_apex_v3/failure_regression.json').read_text())
    display(pd.DataFrame({'original':failure['original_failure_simulation'],
                          'guarded':failure['guarded_failure_simulation']}))
    print('Official V3',failure['official_public_score'],'champion',failure['champion_public_score'])
    print(failure['conclusion'])
    print('Guarded notebook',failure['guarded_notebook'],'eligible',failure['submission_eligible'])
    print('Required next:',failure['required_next'])

V3 fallback regression | estimasi 1s
Official V3 0.055 champion 0.665
A fatal exception is demonstrably swallowed into constant gpt-5.5 predictions in V3. Near-chance LB is consistent with this path; actual hidden exception/logs are unavailable, so its occurrence and underlying model-loading/runtime cause remain unverified.
Guarded notebook notebook/ntl_apex_v3_guarded_UNVALIDATED.ipynb eligible False
Required next: Real full-branch GPU dryrun, inspect mandatory inference logs, then paired quality validation. Do not submit guarded V3 merely because syntax is valid.
V3 fallback regression | selesai 0.01s


,original,guarded
raises,False,True
writes_submission,True,False
unique_labels,1,NaN
synthetic_balanced_accuracy,0.055556,NaN


## 13. E23: depth dan readout checkpoint harus sesuai
Probe CPU pada Docker V3 yang sama menemukan35target modul hilang per adapter
Q25-v5 dengan keep27. Referensi inferensi penulis menyatakan keep32 dan readout
[3,5,8,11,16,21,27,32]. Dimensi delapan readout tetap sama meski indeksnya salah.
Versi2 memeriksa spesifikasi terkoreksi, tanpa memuat bobot backbone besar.
Ini bukan uji CUDA atau bukti kualitas; salinan perbaikan tetap UNVALIDATED.

In [14]:
with step('E23 kontrak checkpoint', 1):
    probe=json.loads((ROOT/'kernels/v3preflight_cpu/out_v2/preflight_report.json').read_text())
    repair=json.loads((ROOT/'research/experiments/e23_checkpoint_preflight/repair_report.json').read_text())
    assert probe['checks']['q25_corrected_contract_passed']
    display(pd.DataFrame([dict(adapter=Path(old['folder']).name,
        submitted_keep=old['source_keep'],missing_submitted=len(old['unresolved_loader_modules']),
        corrected_keep=new['source_keep'],missing_corrected=len(new['unresolved_loader_modules']),
        factor_shape_errors=len(new['shape_mismatches']),head_shape_errors=len(new['head']['format_mismatches']))
        for old,new in zip(probe['checks']['q25'],probe['checks']['q25_corrected_contract'])]))
    print('Author reference',repair['reference']['reference_url'])
    print('Readout',repair['reference']['layers'],'CPU seconds',probe['elapsed_s'])
    print('Fixed notebook',repair['repaired_notebook'],'eligible',repair['submission_eligible'])
    print('Fatal failure simulation',repair['fatal_failure_simulation'])
    print('GPU dryrun',probe['real_gpu_dryrun'],'quality measured: False')

E23 kontrak checkpoint | estimasi 1s
Author reference https://www.kaggle.com/code/dhruvtibarewal/name-that-llm-fully-reproducible
Readout [3, 5, 8, 11, 16, 21, 27, 32] CPU seconds 9.110340885000028
Fixed notebook notebook/ntl_apex_v3_contract_fixed_UNVALIDATED.ipynb eligible False
Fatal failure simulation {'raises': True, 'writes_submission': False}
GPU dryrun False quality measured: False
E23 kontrak checkpoint | selesai 0.01s


,adapter,submitted_keep,missing_submitted,corrected_keep,missing_corrected,factor_shape_errors,head_shape_errors
0,q25_14v5,27,35,32,0,0,0
1,q25_14v5_f0,27,35,32,0,0,0
2,q25_14v5_f1,27,35,32,0,0,0
3,q25_14v5_f2,27,35,32,0,0,0
4,q25_14v5_f3,27,35,32,0,0,0
5,q25_14v5_f4,27,35,32,0,0,0


## 14. E24: prompt-paired column ranking pada frozen head
Bandingkan row-CE biasa dengan row-CE + column-CE di dalam prompt training.
Tidak ada assignment constraint pada test. Lambda dipilih menurut val logloss;
alpha menurut val BA. Gradient/missing-source/offset self-check dan kontrol
label acak di dalam prompt diperiksa. Hasil negatif tetap disimpan.

In [15]:
with step('E24 paired head', 1):
    directory=ROOT/'research/experiments/e24_paired_rank'
    e24=json.loads((directory/'results.json').read_text())
    display(pd.DataFrame({'mean_test':e24['mean_test'],'pooled_test':e24['pooled_test']}).round(5))
    display(pd.DataFrame([dict(comparison=k,ci_low=v[0],ci_high=v[1])
                          for k,v in e24['paired_ci95'].items()]).round(5))
    print('Nonzero lambda selected',e24['selected_nonzero_lambda_folds'],'/5; seconds',e24['elapsed_s'])
    print('Ordinary head lebih baik; tidak dipromosikan atau disubmit.')

E24 paired head | estimasi 1s
Nonzero lambda selected 1 /5; seconds 21.482359874993563
Ordinary head lebih baik; tidak dipromosikan atau disubmit.
E24 paired head | selesai 0.02s


,mean_test,pooled_test
baseline,0.57352,0.57351
ordinary,0.58076,0.58077
paired_selected,0.58028,0.58028
negative_control,0.05705,0.05705


,comparison,ci_low,ci_high
0,selected minus baseline,-0.00483,0.01836
1,selected minus ordinary,-0.00145,0.00000


## 15. E25: StyleDistance sebagai feature gaya murah
Referensi primer: https://aclanthology.org/2025.naacl-long.436/ . Model gaya
bukan detector sumber18LLM. Backbone frozen, official masked-mean pooling,
revision immutable, tanpa remote custom code. Fitur disimpan perbatch dengan
ID dan hash teks. Pilih style/hybrid head menurut val logloss, alpha menurut BA.
Seluruh scaler dilatih hanya pada train. Tidak ada passage terpotong.

In [16]:
with step('E25 style encoder', 1):
    directory=ROOT/'research/experiments/e25_styledistance'
    e25=json.loads((directory/'results.json').read_text())
    feature=json.loads((directory/'feature_manifest.json').read_text())
    extraction=json.loads((directory/'extraction_status.json').read_text())
    assert extraction['complete'] and extraction['rows_done']==2699
    print('Pinned model',feature['model'],feature['revision'])
    print('Metal extraction seconds',extraction['elapsed_s'],'truncated',feature['truncated_rows'])
    records=[json.loads((directory/f'fold{f}.json').read_text()) for f in range(5)]
    display(pd.DataFrame([dict(fold=r['fold'],head=r['chosen']['name'],C=r['chosen']['C'],
        train=r['chosen']['train_ba'],val=r['chosen']['val_ba'],alpha=r['alpha'],**r['test']) for r in records]).round(5))
    print('Test means',e25['mean_test'],'paired CI95',e25['selected_minus_baseline_ci95'])
    print('CPU seconds',e25['elapsed_s'],'not CV of the LB champion; eligible',e25['submission_eligible'])
    print('Base config and pooling config have separate paths in artifact_hash_addendum.json; original manifest retained.')

E25 style encoder | estimasi 1s
Pinned model StyleDistance/styledistance b7df5f0b0480773c097ba3121d83ca32b71015ca
Metal extraction seconds 145.8919189159933 truncated 0
Test means {'baseline': 0.5735177865612648, 'style': 0.3116600790513834, 'selected': 0.5875054896794027} paired CI95 [0.002415458937198068, 0.025612730979901552]
CPU seconds 60.856203625007765 not CV of the LB champion; eligible False
Base config and pooling config have separate paths in artifact_hash_addendum.json; original manifest retained.
E25 style encoder | selesai 0.01s


,fold,head,C,train,val,alpha,baseline,style,selected
0,0,hybrid,0.030,0.85608,0.54365,0.35,0.58454,0.29710,0.61111
1,1,hybrid,0.030,0.86032,0.54798,0.35,0.56324,0.34816,0.59212
2,2,hybrid,0.030,0.85701,0.57323,0.35,0.59662,0.28986,0.60145
3,3,hybrid,0.003,0.68650,0.51515,0.20,0.57246,0.33333,0.57246
4,4,hybrid,0.030,0.85806,0.54545,0.35,0.55072,0.28986,0.56039


## 16. E26: konfirmasi style pada pembanding LoRA yang lebih kuat
Pertahankan comparator/split/alpha0,1 dari E21, ganti fitur menjadi StyleDistance.
Inner96/24 memilih C; refit outer120, test30prompt, semua150prompt tercakup.
E26 memakai style head saja, bukan hybrid E25. Tidak menguji semua kemungkinan
fusion, dan tidak membuktikan bahwa fitur gaya tidak bermanfaat secara umum.

In [17]:
with step('E26 scope confirmation', 1):
    directory=ROOT/'research/experiments/e26_nested_style_confirm'
    e26=json.loads((directory/'results.json').read_text())
    assert e26['test_coverage']==1. and e26['test_rows']==2699
    records=[json.loads((directory/f'fold{f}.json').read_text()) for f in range(5)]
    display(pd.DataFrame([dict(fold=r['fold'],C=r['chosen']['C'],inner_train=r['chosen']['train_ba'],
        inner_val=r['chosen']['val_ba'],refit_train=r['train_refit_ba'],**r['test']) for r in records]).round(5))
    print('Means',e26['mean_test'],'paired CI95',e26['paired_ci95'])
    print('Positive folds',e26['positive_folds'],'/5; CPU seconds',e26['elapsed_s'])
    print('CI melintasi0; tidak dipromosikan. No new competition submission; official best0.665.')

E26 scope confirmation | estimasi 1s
Means {'baseline': 0.597088122605364, 'fusion': 0.5985696040868455, 'reader': 0.31415070242656445} paired CI95 [-0.005548060815917959, 0.008158153119562514]
Positive folds 2 /5; CPU seconds 40.21808449999662
CI melintasi0; tidak dipromosikan. No new competition submission; official best0.665.
E26 scope confirmation | selesai 0.01s


,fold,C,inner_train,inner_val,refit_train,baseline,reader,fusion
0,0,0.030,0.69363,0.30787,0.68361,0.62037,0.34259,0.62037
1,1,0.030,0.71737,0.29630,0.67943,0.60741,0.36111,0.61667
2,2,0.030,0.70833,0.29861,0.68287,0.57433,0.30409,0.57803
3,3,0.003,0.43827,0.26157,0.43301,0.60185,0.26852,0.59815
4,4,0.030,0.69769,0.32870,0.68499,0.58148,0.29444,0.57963
